In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
print(df.info())
print("\nMissing values:")
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   checking_status         1000 non-null   category
 1   duration                1000 non-null   int64   
 2   credit_history          1000 non-null   category
 3   purpose                 1000 non-null   category
 4   credit_amount           1000 non-null   int64   
 5   savings_status          1000 non-null   category
 6   employment              1000 non-null   category
 7   installment_commitment  1000 non-null   int64   
 8   personal_status         1000 non-null   category
 9   other_parties           1000 non-null   category
 10  residence_since         1000 non-null   int64   
 11  property_magnitude      1000 non-null   category
 12  age                     1000 non-null   int64   
 13  other_payment_plans     1000 non-null   category
 14  housing                 1

In [ ]:
from sklearn.datasets import fetch_openml

credit = fetch_openml("credit-g", version=1, as_frame=True)

df = credit.frame

print(df.head())
print("Dataset shape:", df.shape)

  checking_status  duration                  credit_history  \
0              <0         6  critical/other existing credit   
1        0<=X<200        48                   existing paid   
2     no checking        12  critical/other existing credit   
3              <0        42                   existing paid   
4              <0        24              delayed previously   

               purpose  credit_amount    savings_status employment  \
0             radio/tv           1169  no known savings        >=7   
1             radio/tv           5951              <100     1<=X<4   
2            education           2096              <100     4<=X<7   
3  furniture/equipment           7882              <100     4<=X<7   
4              new car           4870              <100     1<=X<4   

   installment_commitment     personal_status other_parties  ...  \
0                       4         male single          none  ...   
1                       2  female div/dep/mar          none  ...

In [ ]:
X = df.drop("class", axis=1)
y = df["class"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (1000, 20)
Target shape: (1000,)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (800, 20)
Testing data: (200, 20)


In [ ]:
categorical_cols = X.select_dtypes(include=["category"]).columns
numerical_cols = X.select_dtypes(include=["int64"]).columns

print("Categorical columns:", len(categorical_cols))
print("Numerical columns:", len(numerical_cols))

Categorical columns: 13
Numerical columns: 7


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Preprocessing complete!")
print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)

Preprocessing complete!
Training shape: (800, 61)
Testing shape: (200, 61)


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )
}

for name, model in models.items():
    model.fit(X_train_processed, y_train)
    print(name, "trained successfully!")

Logistic Regression trained successfully!
Decision Tree trained successfully!
Random Forest trained successfully!


In [ ]:
for name, model in models.items():
    y_pred = model.predict(X_test_processed)
    y_prob = model.predict_proba(X_test_processed)[:, 1]

    print("\n", name)
    print("Precision:", precision_score(y_test, y_pred, pos_label="good"))
    print("Recall:", recall_score(y_test, y_pred, pos_label="good"))
    print("F1-Score:", f1_score(y_test, y_pred, pos_label="good"))
    print("ROC-AUC:", roc_auc_score(
        (y_test == "good").astype(int),
        (y_prob if list(model.classes_) == ["bad", "good"] else 1 - y_prob)
    ))


 Logistic Regression
Precision: 0.7755102040816326
Recall: 0.8142857142857143
F1-Score: 0.794425087108014
ROC-AUC: 0.7594047619047619

 Decision Tree
Precision: 0.7518248175182481
Recall: 0.7357142857142858
F1-Score: 0.7436823104693141
ROC-AUC: 0.5845238095238096

 Random Forest
Precision: 0.7987012987012987
Recall: 0.8785714285714286
F1-Score: 0.8367346938775511
ROC-AUC: 0.7799404761904762


In [ ]:
import joblib

best_model = models["Random Forest"]

joblib.dump(best_model, "random_forest_credit_model.pkl")
joblib.dump(preprocessor, "credit_preprocessor.pkl")

print("Best model saved successfully!")

Best model saved successfully!


In [ ]:
print("Final Model: Random Forest")
print("Precision: 0.799")
print("Recall: 0.879")
print("F1-Score: 0.837")
print("ROC-AUC: 0.780")

Final Model: Random Forest
Precision: 0.799
Recall: 0.879
F1-Score: 0.837
ROC-AUC: 0.780
